# Previsão de Demanda (Forecast de Vendas)

**Objetivo de Negócio:** Prever o volume de vendas para os próximos meses a fim de otimizar os níveis de estoque e a política de compras (Purchase Orders), evitando rupturas (falta de produto) durante a alta temporada.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing

sns.set_theme(style="whitegrid")
con = duckdb.connect()
print("Bibliotecas importadas com sucesso.")

## 1. Extração e Preparação da Série Temporal

Vamos extrair a receita mensal de 2020 até Julho de 2026. (Omitimos Agosto de 2026 pois, como vimos na auditoria, os dados vão apenas até o dia 10, o que causaria uma queda artificial na série).

In [ ]:
query = """
    SELECT 
        DATE_TRUNC('month', placed_at) as mes,
        SUM(total) as receita
    FROM read_csv_auto('E:/repo/lh_nautical_analise/data/raw/orders.csv')
    WHERE placed_at >= '2020-01-01' AND placed_at < '2026-08-01'
    GROUP BY 1
    ORDER BY 1
"""
df_ts = con.execute(query).df()
df_ts['mes'] = pd.to_datetime(df_ts['mes'])
df_ts.set_index('mes', inplace=True)
df_ts.index.freq = 'MS'

plt.figure(figsize=(12, 5))
plt.plot(df_ts.index, df_ts['receita'], marker='o')
plt.title('Receita Mensal Histórica (2020 - Julho 2026)')
plt.ylabel('Receita')
plt.show()

## 2. Decomposição Sazonal (O que explica o sobe-e-desce?)

Como a Diretoria (Sr. Almir) não gosta de "caixas pretas", antes de prever, vamos separar a série em Tendência (Crescimento de longo prazo) e Sazonalidade (Padrão repetitivo anual).

In [ ]:
decomposicao = seasonal_decompose(df_ts['receita'], model='multiplicative')

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10))
ax1.plot(df_ts.index, df_ts['receita'], label='Original')
ax1.plot(df_ts.index, decomposicao.trend, label='Tendência (Crescimento)', color='red')
ax1.legend()

ax2.plot(decomposicao.seasonal, color='green')
ax2.set_title('Sazonalidade (Fator Multiplicativo Anual)')

ax3.plot(decomposicao.resid, color='gray')
ax3.set_title('Resíduos (Ruído)')

plt.tight_layout()
plt.show()

## 3. Modelo Preditivo Explicável (Holt-Winters)

Usaremos o método de Suavização Exponencial Tripla (Holt-Winters) pois ele lida perfeitamente com sazonalidade e tendência, sem a complexidade obscura de redes neurais.
Vamos projetar a demanda para os próximos 6 meses (Agosto 2026 - Janeiro 2027).

In [ ]:
# Treinando o modelo Holt-Winters (Sazonalidade aditiva de 12 meses)
modelo_hw = ExponentialSmoothing(
    df_ts['receita'], 
    trend='add', 
    seasonal='add', 
    seasonal_periods=12
).fit()

# Previsão para 6 meses
forecast = modelo_hw.forecast(6)

plt.figure(figsize=(12, 5))
plt.plot(df_ts.index[-24:], df_ts['receita'][-24:], label='Histórico Recente (Últimos 2 anos)')
plt.plot(forecast.index, forecast, label='Previsão (Ago/26 - Jan/27)', color='red', linestyle='--')
plt.legend()
plt.title('Previsão de Demanda para a Próxima Alta Temporada')
plt.show()

forecast.to_frame('Receita_Prevista')

## 4. Insight de Negócio (Framework F-H-R)

**Fato Observado:** A LH Nautical possui um crescimento orgânico contínuo (Year-over-Year) e uma sazonalidade fortíssima, com picos massivos de venda sempre no Verão (Dezembro a Fevereiro) e vales no meio do ano (Junho/Julho).

**Hipótese:** Sendo uma loja de artigos náuticos, os consumidores antecipam e realizam compras no pico do verão e férias. O crescimento YOY indica maturidade e expansão de *market share* ou da própria base inflacionada de clientes identificada na etapa 04.

**Recomendação:** A área de Compras (Suprimentos) deve emitir *Purchase Orders* robustas imediatamente (agora, em Agosto/Setembro) para garantir estoques adequados entre Novembro e Janeiro (que baterão novos recordes, ultrapassando R$ 40 milhões em Janeiro/27, segundo a previsão). Se não houver reabastecimento logístico nos próximos 60 dias, a empresa sofrerá ruptura de estoque grave na alta temporada.